# MCP 入門工作坊 — Hands-on（Colab 版）

> NAPAI · SIGAgent · MCP 工作坊 · Segment 4

**不用在自己電腦裝任何東西。** 每個格子左邊有 ▶，從上到下一個一個按。

跑完你會：
1. 在瀏覽器裡得到一個會自己選工具的 AI 助理（完整 web 介面）
2. 把它改成 **你自己領域** 的助理（換資料、改說明書，不寫一行新邏輯）

⏱ 約 10 分鐘環境 + 30 分鐘 L1 練習

## Section 1 · 環境準備（約 3 分鐘，跑一次就好）

In [ ]:
# === 1. 下載教材 + 裝 Node 20 與 uv ===
import os
if not os.path.exists('nchu-mcp-workshop-2026'):
    !git clone -q https://github.com/UDICatNCHU/nchu-mcp-workshop-2026
%cd nchu-mcp-workshop-2026/mini-project

# Colab 預設 Node 太舊,裝 Node 20（NodeSource）
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1

# uv：Python 套件管理,MCP 工具伺服器用
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

!node --version
!uv --version

In [ ]:
# === 2. 安裝相依套件（約 1 分鐘）===
!cd backend-node && npm install --silent
!cd mcp-server-py && uv sync --quiet
print('✅ 依賴安裝完成')

In [ ]:
# === 3. 貼上你的 Claude API key（講師會發給你）===
import os, getpass
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('貼上 Claude API key（sk-ant-...）後按 Enter: ').strip()
os.environ['LLM_PROVIDER'] = 'claude'
assert os.environ['ANTHROPIC_API_KEY'].startswith('sk-ant-'), '⚠ key 格式怪怪的,應以 sk-ant- 開頭'
print('✅ key 已設定')

## Section 2 · 啟動助理

下面這格啟動 backend（它會自動開出 Python MCP 工具子程序）。
**L1 改完檔案後，回來重跑這一格就能套用改動。**

In [ ]:
# === 4. 啟動 / 重啟 server ===
import subprocess, time, os

_server = None
def start_server():
    global _server
    if _server and _server.poll() is None:
        _server.terminate()
        try: _server.wait(timeout=5)
        except Exception: _server.kill()
        time.sleep(1)
    logf = open('/content/server.log', 'w')
    _server = subprocess.Popen(
        ['node', 'server.js'], cwd='backend-node',
        env=os.environ, stdout=logf, stderr=subprocess.STDOUT, text=True,
    )
    for _ in range(25):
        time.sleep(1)
        try: log = open('/content/server.log').read()
        except FileNotFoundError: log = ''
        if 'Mini AI Assistant' in log:
            print('✅ server 啟動成功')
            for line in log.splitlines():
                if '✓' in line:
                    print('  ', line.strip())
            return
    print('⚠ server 還沒就緒,最後 log：')
    print(open('/content/server.log').read()[-1200:])

start_server()

## Section 3 · 打開聊天介面

下面這格會把聊天介面 **直接顯示在格子下方**。
試問：「英文中心幾點開門？」→ 看它呼叫工具回答你。

In [ ]:
# === 5A. 把聊天介面顯示在下方（主要方式）===
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(3000, height=620)

### 萬一上面沒出現介面 / 卡住？

跑下面這格，用 cloudflared 開一個 **公開網址**（新分頁全螢幕，零帳號）。
只在上面那格不動時才需要跑。

In [ ]:
# === 5B. 備援：cloudflared 公開網址 ===
import subprocess, time, re
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://localhost:3000'],
                 stdout=open('/content/tunnel.log','w'),
                 stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(25):
    time.sleep(1)
    try: m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    except FileNotFoundError: m = None
    if m: url = m.group(0); break
print('🌐 在新分頁打開：', url or '(還沒拿到 URL,再跑一次這格)')

## 【L1】把它改成你自己領域的助理（課堂主練習）

現在這個助理只懂「英文中心」。接下來 **不寫一行新 Python 邏輯**，
把它改成你的：實驗室介紹 / 課程問答 / 研究成果 / 任何你想被自然語言問的資料。

三步：① 換資料 → ② 改說明書(docstring) → ③ 重啟 + 驗證

In [ ]:
# === L1 ① 換成你的資料 ===
# 改成你自己領域的內容,然後按 ▶
import json, pathlib

MY_DATA = {
    "lab_name": "XXX 教授 NLP 研究室",          # ← 改這裡
    "pi": {"name": "XXX", "email": "xxx@nchu.edu.tw", "office": "資電 520"},
    "research_areas": ["大型語言模型評測", "對話系統"],
    "recent_publications": [
        {"year": 2025, "venue": "ACL", "title": "..."},
    ],
    "openings": {"master": 2, "phd": 1, "prerequisites": "修過機器學習"},
    "weekly_meeting": "每週四 14:00 資電 601",
}

p = pathlib.Path('mcp-server-py/data/english_center.json')
p.write_text(json.dumps(MY_DATA, ensure_ascii=False, indent=2), encoding='utf-8')
print('✅ 資料已寫入', p)

In [ ]:
# === L1 ② 改 docstring（tool 對 LLM 的「說明書」）===
# LLM 靠這段文字決定「要不要呼叫這個工具」。寫清楚『使用情境』很重要。
TOOL_DOCSTRING = """取得 XXX 教授 NLP 研究室完整資訊。

使用情境：使用者詢問研究室的任何資訊,包含研究方向、
PI 聯絡方式、近期論文、招生名額與門檻、週會時間時呼叫。

回傳 JSON 字串。"""

# 把新 docstring 寫回 hello_tool.py（其餘程式不動）
import re, pathlib
path = pathlib.Path('mcp-server-py/hello_tool.py')
src = path.read_text(encoding='utf-8')
new_func = (
    "@mcp.tool()\n"
    "def get_english_center_info() -> str:\n"
    '    """' + TOOL_DOCSTRING + '"""\n'
    "    if not INFO:\n"
    '        return json.dumps({"error": "資料檔不存在"}, ensure_ascii=False)\n'
    "    return json.dumps(INFO, ensure_ascii=False, indent=2)"
)
src = re.sub(r'@mcp\.tool\(\)\ndef get_english_center_info.*?indent=2\)',
             new_func, src, flags=re.DOTALL)
path.write_text(src, encoding='utf-8')
print('✅ docstring 已更新')

In [ ]:
# === L1 ③ 重啟 server 讓改動生效,再打開介面 ===
start_server()
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(3000, height=620)
# 試問你自己的資料,例如:「研究室在研究什麼?」「PI 的 email?」「碩士班有名額嗎?」

### ✅ 驗證（做完代表 L1 完成）

- [ ] 問「研究室在研究什麼？」→ 答得出你 JSON 裡的內容
- [ ] 問「今天天氣？」→ AI **不會** 亂呼叫你的工具（它知道不相關）
- [ ] 問一個你資料裡 **沒有** 的東西（例如「有實習嗎？」）→ AI 老實說不知道，**不編造**

最後一項最關鍵 —— tool-calling 正確運作的標誌是「沒有的東西不會胡編」。

## 你剛剛做了什麼？

- 跑起一個 **完整三層** 的 AI 助理：Python MCP 工具 ↔ Node 後端 ↔ web 介面
- 改資料 + 改 docstring，就讓它變成你領域的助理 —— **沒寫一行新邏輯**

### 課後路徑
- **L2** 加一支有參數的搜尋工具 → `docs/labs/L2-add-a-search-tool.md`
- **L3** 呼叫外部 API → `docs/labs/L3-call-external-api.md`
- **完整 web 版**（在自己電腦跑）→ `mini-project/README.md`